# 🏠 Housing Price Prediction — Lasso Regression
This notebook trains a **Lasso Regression** model (L1 regularization) on the California Housing dataset.  
Uses a **Pipeline** with `OneHotEncoder` + `StandardScaler` — same approach as the AQI Lasso model.  
Upload your `housing.csv` from your local machine using the file picker below.

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
!pip install -q scikit-learn pandas numpy matplotlib seaborn joblib

In [ ]:
# ── Upload dataset from local machine ────────────────────────────────────────
from google.colab import files

print('📂 Please select your housing.csv file...')
uploaded = files.upload()
filename = list(uploaded.keys())[0]
print(f'✅ Uploaded: {filename}')

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import io
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style='whitegrid', palette='muted')
print('Libraries loaded ✅')

In [ ]:
# ── Load & preview data ───────────────────────────────────────────────────────
df = pd.read_csv(io.BytesIO(uploaded[filename]))
print('Shape:', df.shape)
df.head()

In [ ]:
# ── Basic info & missing values ───────────────────────────────────────────────
print('=== Data Types ===')
print(df.dtypes)
print('\n=== Missing Values ===')
print(df.isnull().sum())

In [ ]:
# ── Preprocessing ─────────────────────────────────────────────────────────────
df = df.dropna()

# Numeric + categorical feature columns
num_cols = [
    'longitude', 'latitude', 'housing_median_age',
    'total_rooms', 'total_bedrooms', 'population',
    'households', 'median_income'
]
cat_cols = ['ocean_proximity']

X = df[num_cols + cat_cols]
y = df['median_house_value']

print('Features:', num_cols + cat_cols)
print('Target: median_house_value')

In [ ]:
# ── Train / Test split ────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print(f'Train size: {X_train.shape[0]}  |  Test size: {X_test.shape[0]}')

In [ ]:
# ── Build Pipeline (same structure as AQI Lasso model) ───────────────────────
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ]
)

model = Pipeline([
    ('pre', preprocessor),
    ('lasso', Lasso(alpha=0.1))
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_test, y_pred)

print(f'MSE  : {mse:,.2f}')
print(f'RMSE : {rmse:,.2f}')
print(f'R²   : {r2:.4f}')

## 📊 Visualizations

In [ ]:
# ── 1. Actual vs Predicted ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(y_test, y_pred, alpha=0.3, edgecolors='k', linewidths=0.4, color='darkorange')
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ax.plot(lims, lims, 'r--', linewidth=2, label='Perfect prediction')
ax.set_xlabel('Actual House Value ($)', fontsize=12)
ax.set_ylabel('Predicted House Value ($)', fontsize=12)
ax.set_title('Lasso Regression — Actual vs Predicted', fontsize=14)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── 2. Residuals Distribution ────────────────────────────────────────────────
residuals = y_test.values - y_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(residuals, bins=60, color='darkorange', edgecolor='white')
axes[0].axvline(0, color='red', linestyle='--')
axes[0].set_title('Residuals Distribution', fontsize=13)
axes[0].set_xlabel('Residual')
axes[0].set_ylabel('Count')

axes[1].scatter(y_pred, residuals, alpha=0.3, color='darkorange', edgecolors='k', linewidths=0.3)
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_title('Residuals vs Fitted Values', fontsize=13)
axes[1].set_xlabel('Fitted Values')
axes[1].set_ylabel('Residuals')

plt.tight_layout()
plt.show()

In [ ]:
# ── 3. Lasso Feature Coefficients (zero = feature eliminated) ────────────────
lasso_step = model.named_steps['lasso']
pre_step   = model.named_steps['pre']

# Reconstruct feature names after preprocessing
num_feature_names = num_cols
cat_feature_names = list(
    pre_step.named_transformers_['cat'].get_feature_names_out(cat_cols)
)
all_feature_names = num_feature_names + cat_feature_names

coef_df = pd.DataFrame({
    'Feature': all_feature_names,
    'Coefficient': lasso_step.coef_
}).sort_values('Coefficient')

colors = ['tomato' if c < 0 else 'darkorange' for c in coef_df['Coefficient']]

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Lasso Regression — Feature Coefficients\n(zero = feature eliminated by L1)', fontsize=13)
ax.set_xlabel('Coefficient Value')
plt.tight_layout()
plt.show()

In [ ]:
# ── 4. Lasso alpha tuning — RMSE vs alpha ────────────────────────────────────
alphas = [0.001, 0.01, 0.1, 1, 10, 100, 1000]
rmse_scores = []

for a in alphas:
    pipe = Pipeline([
        ('pre', preprocessor),
        ('lasso', Lasso(alpha=a))
    ])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    rmse_scores.append(np.sqrt(mean_squared_error(y_test, pred)))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(alphas, rmse_scores, marker='o', color='darkorange', linewidth=2)
ax.axvline(0.1, color='red', linestyle='--', label='Current alpha=0.1')
ax.set_xscale('log')
ax.set_title('RMSE vs Lasso Alpha', fontsize=13)
ax.set_xlabel('Alpha (log scale)')
ax.set_ylabel('RMSE')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── 5. Correlation Heatmap ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))
corr = df[num_cols + ['median_house_value']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax,
            square=True, linewidths=0.5)
ax.set_title('Feature Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── Save model ────────────────────────────────────────────────────────────────
joblib.dump(model, 'housing_lasso_model.pkl')
print('Model saved as housing_lasso_model.pkl')
files.download('housing_lasso_model.pkl')